# Reconstructing a HELENA input from a data sample

This notebook demonstrates how to reconstruct a `fort.10` input file from a
sample HDF5 file.

The reconstructed file can be used to rerun HELENA, making each HDF5 sample a
self-contained representation of an equilibrium.

In [2]:
from pathlib import Path
import f90nml

from karhu_data_handling.convert_sample import recreate_fort10

In [13]:
sample_file = Path("..\\data\\sample.h5")
original_fort10 = Path("..\\data\\HelenaRunner-0a8221be-38f7-4679-a937-566e6bf83d5a_scan_2\\fort.10")

output_fort10 = Path("fort.10")

In [4]:
recreate_fort10(sample_file, output_fort10)

fort.10 written to fort.10


In [5]:
nml = f90nml.read(output_fort10)

print("Namelists:")
for name in nml:
    print(f"  {name}")

Namelists:
  ball
  num
  phys
  plot
  pri
  profile
  shape


In [8]:
import h5py

with h5py.File(sample_file, "r") as h5:
    inputs = h5["equilibrium"]["inputs"]
    print("Stored namelists:")

    for name in inputs:
        print(f"  {name}")

Stored namelists:
  ball
  num
  phys
  plot
  pri
  profile
  shape


In [ ]:
from pathlib import Path
import f90nml
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


def flatten_namelist(nml, prefix=""):
    """
    Flatten a nested f90nml.Namelist into a dictionary.

    Keys are of the form:
        physics.beta
        shape.tria
        ...
    """
    result = {}

    for group_name, group in nml.items():
        for key, value in group.items():
            result[f"{group_name}.{key}"] = value

    return result


original = flatten_namelist(f90nml.read(original_fort10))
recreated = flatten_nml = flatten_namelist(f90nml.read(output_fort10))

# Union of all keys
keys = sorted(set(original) | set(recreated))

rows = []

for key in keys:
    rows.append({
        "Parameter": key,
        "Original": original.get(key),
        "Recreated": recreated.get(key),
        "Match": original.get(key) == recreated.get(key)
    })

comparison = pd.DataFrame(rows)

display(comparison)

,Parameter,Original,Recreated,Match
0,num.amix,0.0,0.0,True
1,num.errcur,0.0,0.0,True
2,num.nchi,513,513,True
3,num.niter,50,50,True
4,num.nmesh,100,100,True
5,num.np,129,129,True
6,num.npcur,33,33,True
7,num.npmap,513,513,True
8,num.nr,301,301,True
9,num.nrcur,150,150,True


The reconstructed `fort.10` is functionally equivalent to the original input
file. While formatting, comments, and whitespace may differ, all namelists and
parameter values are preserved in the HDF5 sample.

This enables complete reconstruction of the HELENA input from the archived data
without requiring access to the original simulation directory.